In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="sctp-team2-project2-elt")

query = """
SELECT
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.order_status,
    r.review_score,
    c.customer_unique_id,

    -- delivery delay in days (positive = late, negative = early)
    DATE_DIFF(
        DATE(o.order_delivered_customer_date),
        DATE(o.order_estimated_delivery_date),
        DAY
    ) AS delivery_delay_days

FROM `sctp-team2-project2-elt.olist_gold_mart_prod.fact_orders` f
JOIN `sctp-team2-project2-elt.olist_bronze_prod.olist_orders_raw` o 
    ON f.order_id = o.order_id
JOIN `sctp-team2-project2-elt.olist_bronze_prod.olist_order_reviews_raw` r 
    ON o.order_id = r.order_id
JOIN `sctp-team2-project2-elt.olist_bronze_prod.olist_customers_raw` c 
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
"""

df = client.query(query).to_dataframe()
df.head()